In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
MDEB_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/02-transformer-training")

CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}

train = pd.read_csv(DATA_DIR / "train.csv")
splits = pd.read_parquet(BASE_DIR / "folds.parquet")
base_oof = pd.read_parquet(BASE_DIR / "baseline_oof_fold0.parquet")
mdeb_oof = pd.read_parquet(MDEB_DIR / "mdeberta_oof_fold0.parquet")

assert (train["id"].to_numpy() == splits["id"].to_numpy()).all()
train["fold"] = splits["fold"].to_numpy()
train["y"] = train["label"].map(LABEL2ID).astype(np.int64)

va = train[train["fold"] == 0].reset_index(drop=True)
assert (va["id"].to_numpy() == base_oof["id"].to_numpy()).all(), "Baseline OOF ID mismatch!"
assert (va["id"].to_numpy() == mdeb_oof["id"].to_numpy()).all(), "mDeBERTa OOF ID mismatch!"
assert (va["y"].to_numpy() == mdeb_oof["y_true"].to_numpy()).all(), "Label mapping mismatch!"

print("Baseline OOF cols:", list(base_oof.columns))
print("mDeBERTa OOF cols:", list(mdeb_oof.columns))

va["pred_base"] = base_oof["y_pred"].to_numpy()
va["pred_mdeb"] = mdeb_oof["y_pred"].to_numpy()
va["n_words"] = va["text"].str.split().str.len()

# 1. Confusion Matrix (mDeBERTa)
cm = pd.DataFrame(
    confusion_matrix(va["y"], va["pred_mdeb"], labels=[0, 1, 2]),
    index=[f"true_{c}" for c in CLASSES],
    columns=[f"pred_{c}" for c in CLASSES],
)
print("\n--- Confusion Matrix (mDeBERTa, Val Fold 0) ---")
print(cm)

# 2. Взаємодоповнюваність із TF-IDF
mdeb_err = va["pred_mdeb"] != va["y"]
base_ok = va["pred_base"] == va["y"]
print(
    f"\nmDeBERTa errors: {mdeb_err.sum()} ({mdeb_err.mean()*100:.2f}%) | "
    f"of which TF-IDF is correct: {(mdeb_err & base_ok).sum()} "
    f"({(mdeb_err & base_ok).sum() / mdeb_err.sum() * 100:.1f}%)"
)

def group_metrics(df, col):
    rows = []
    for k, g in df.groupby(col, observed=False):
        y, p = g["y"].to_numpy(), g["pred_mdeb"].to_numpy()
        macro = f1_score(y, p, average="macro", zero_division=0)
        per = f1_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
        f_to_h = ((y == 1) & (p == 0)).sum() / max(1, (y == 1).sum()) * 100
        h_to_f = ((y == 0) & (p == 1)).sum() / max(1, (y == 0).sum()) * 100
        rows.append({
            col: k, "n": len(g), "macro_f1": macro,
            "f1_H": per[0], "f1_F": per[1], "f1_A": per[2],
            "F->H %": f_to_h, "H->F %": h_to_f,
        })
    return pd.DataFrame(rows).round(4)

# 3. Розбивка по мові, домену та довжині (~350 слів ≈ 512 токенів)
lang_col = next(c for c in ("language", "lang") if c in va.columns)
print(f"\n--- Metrics by {lang_col} ---")
print(group_metrics(va, lang_col).to_string(index=False))

print("\n--- Metrics by domain ---")
print(group_metrics(va, "domain").sort_values("macro_f1").to_string(index=False))

va["len_bin"] = pd.cut(
    va["n_words"],
    bins=[0, 200, 350, 550, 900, 10**9],
    labels=["<=200w", "201-350w (~512t)", "351-550w", "551-900w", ">900w"],
)
print("\n--- Metrics by text length (words) ---")
print(group_metrics(va, "len_bin").to_string(index=False))

Baseline OOF cols: ['id', 'y_true', 'y_pred', 'score_human', 'score_fusion', 'score_ai']
mDeBERTa OOF cols: ['id', 'y_true', 'y_pred', 'logit_human', 'logit_fusion', 'logit_ai']

--- Confusion Matrix (mDeBERTa, Val Fold 0) ---
             pred_human  pred_fusion  pred_ai
true_human        11920         1417       52
true_fusion        1359        15556       53
true_ai              21           41    24171

mDeBERTa errors: 2943 (5.39%) | of which TF-IDF is correct: 1905 (64.7%)

--- Metrics by lang ---
 lang     n  macro_f1   f1_H   f1_F   f1_A  F->H %  H->F %
de-DE 10330    0.9211 0.8644 0.9034 0.9957   5.116 18.4118
en-EN 44260    0.9387 0.9005 0.9190 0.9967   8.782  8.4701

--- Metrics by domain ---
                domain     n  macro_f1   f1_H   f1_F   f1_A  F->H %  H->F %
             gutenberg  1529    0.7678 0.6038 0.7020 0.9975 17.6471 45.1613
          arxiv_papers  2850    0.7961 0.4746 0.9138 0.9998  2.0468 66.2651
      euro_court_cases  5575    0.8709 0.7652 0.8475 1.000

In [2]:
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ID2LABEL = {i: c for i, c in enumerate(CLASSES)}

def extract_scores(df):
    for prefix in ("logit_", "score_", "dec_", "prob_", "p_"):
        cols = [f"{prefix}{c}" for c in CLASSES]
        if all(c in df.columns for c in cols):
            return df[cols].to_numpy(dtype=np.float64), cols
    cols = [
        next(c for c in df.columns if cls in c.lower() and c not in ("id", "label", "y_true", "y_pred"))
        for cls in CLASSES
    ]
    return df[cols].to_numpy(dtype=np.float64), cols

def fast_macro_f1(y_true, y_pred):
    cm = np.bincount(y_true * 3 + y_pred, minlength=9).reshape(3, 3)
    tp = np.diag(cm)
    denom = cm.sum(axis=0) + cm.sum(axis=1)
    return float(np.mean(np.where(denom > 0, 2.0 * tp / denom, 0.0)))

def fit_multipliers(probs, y_true):
    # w_ai фіксуємо = 1.0 (масштабно-інваріантно для argmax)
    best_f1, best_w = -1.0, (1.0, 1.0)
    for w0 in np.linspace(0.5, 1.6, 23):
        for w1 in np.linspace(0.5, 1.6, 23):
            preds = (probs * np.array([w0, w1, 1.0])).argmax(axis=1)
            score = fast_macro_f1(y_true, preds)
            if score > best_f1:
                best_f1, best_w = score, (w0, w1)
    # Точний пошук навколо оптимуму
    w0_c, w1_c = best_w
    for w0 in np.linspace(max(0.2, w0_c - 0.1), w0_c + 0.1, 21):
        for w1 in np.linspace(max(0.2, w1_c - 0.1), w1_c + 0.1, 21):
            preds = (probs * np.array([w0, w1, 1.0])).argmax(axis=1)
            score = fast_macro_f1(y_true, preds)
            if score > best_f1:
                best_f1, best_w = score, (w0, w1)
    return np.array([best_w[0], best_w[1], 1.0])

base_te = pd.read_parquet(BASE_DIR / "baseline_test_preds.parquet")
mdeb_te = pd.read_parquet(MDEB_DIR / "mdeberta_test_preds.parquet")
assert (base_te["id"].to_numpy() == mdeb_te["id"].to_numpy()).all(), "Test ID mismatch!"

S_base_va, base_cols = extract_scores(base_oof)
S_base_te, _ = extract_scores(base_te)
L_mdeb_va, mdeb_cols = extract_scores(mdeb_oof)
L_mdeb_te, _ = extract_scores(mdeb_te)
print(f"Detected score cols | Baseline: {base_cols} | mDeBERTa: {mdeb_cols}")

y_va = va["y"].to_numpy()
P_mdeb_va = softmax(L_mdeb_va, axis=1)
X_meta_va = np.hstack([L_mdeb_va, S_base_va])
X_meta_te = np.hstack([L_mdeb_te, S_base_te])

# Внутрішній 5-fold CV на fold 0 (без витоку)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_mdeb_thr = np.zeros(len(y_va), dtype=np.int64)
oof_blend_prob = np.zeros((len(y_va), 3), dtype=np.float64)
oof_blend_thr = np.zeros(len(y_va), dtype=np.int64)

te_blend_prob = np.zeros((len(mdeb_te), 3), dtype=np.float64)
fold_mults_mdeb, fold_mults_blend = [], []

for tr_idx, ev_idx in skf.split(X_meta_va, y_va):
    # A. Лише тюнінг порогів для mDeBERTa
    w_m = fit_multipliers(P_mdeb_va[tr_idx], y_va[tr_idx])
    fold_mults_mdeb.append(w_m)
    oof_mdeb_thr[ev_idx] = (P_mdeb_va[ev_idx] * w_m).argmax(axis=1)

    # B. Мета-модель (LogReg поверх логітів mDeBERTa + скорів TF-IDF)
    meta = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=1000))
    meta.fit(X_meta_va[tr_idx], y_va[tr_idx])
    p_tr = meta.predict_proba(X_meta_va[tr_idx])
    p_ev = meta.predict_proba(X_meta_va[ev_idx])
    oof_blend_prob[ev_idx] = p_ev
    te_blend_prob += meta.predict_proba(X_meta_te) / skf.n_splits

    # C. Бленд + тюнінг порогів
    w_b = fit_multipliers(p_tr, y_va[tr_idx])
    fold_mults_blend.append(w_b)
    oof_blend_thr[ev_idx] = (p_ev * w_b).argmax(axis=1)

results = [
    ("1. TF-IDF baseline", va["pred_base"].to_numpy()),
    ("2. mDeBERTa raw (ref)", va["pred_mdeb"].to_numpy()),
    ("3. mDeBERTa + multipliers (CV)", oof_mdeb_thr),
    ("4. mDeBERTa + TF-IDF blend (CV)", oof_blend_prob.argmax(axis=1)),
    ("5. Blend + multipliers (CV)", oof_blend_thr),
]

rows = []
for name, pred in results:
    per = f1_score(y_va, pred, average=None, labels=[0, 1, 2])
    rows.append({
        "model": name,
        "macro_f1": f1_score(y_va, pred, average="macro"),
        "f1_H": per[0], "f1_F": per[1], "f1_A": per[2],
    })
res_df = pd.DataFrame(rows).round(4)
print("\n--- Validation Fold 0 Comparison (Nested CV, no leakage) ---")
print(res_df.to_string(index=False))

mean_w_b = np.mean(fold_mults_blend, axis=0)
print(f"\nMean blend multipliers [H, F, A]: {np.round(mean_w_b, 4)}")

# Збереження сабмішну, якщо найкращий варіант перевищує референс
best_row = res_df.iloc[res_df["macro_f1"].idxmax()]
ref_f1 = res_df.loc[res_df["model"] == "2. mDeBERTa raw (ref)", "macro_f1"].iloc[0]

if best_row["macro_f1"] > ref_f1:
    sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
    if "Blend + multipliers" in best_row["model"]:
        te_preds = (te_blend_prob * mean_w_b).argmax(axis=1)
    elif "blend" in best_row["model"].lower():
        te_preds = te_blend_prob.argmax(axis=1)
    else:
        te_preds = (softmax(L_mdeb_te, axis=1) * np.mean(fold_mults_mdeb, axis=0)).argmax(axis=1)

    pred_map = pd.Series(te_preds, index=mdeb_te["id"]).map(ID2LABEL)
    sub = pd.DataFrame({"id": sample_sub["id"], "label": sample_sub["id"].map(pred_map)})
    assert len(sub) == 25_000 and (sub["id"].to_numpy() == sample_sub["id"].to_numpy()).all()
    assert sub["label"].notna().all() and set(sub["label"].unique()) == set(CLASSES)
    sub.to_csv("/kaggle/working/submission_blend_mdeb_tfidf.csv", index=False)
    print(f"\nSaved submission_blend_mdeb_tfidf.csv (best: {best_row['model']}, +{best_row['macro_f1'] - ref_f1:.4f} vs ref)")
    print((sub["label"].value_counts(normalize=True) * 100).round(2).reindex(CLASSES))
else:
    print("\nNo improvement over mDeBERTa raw — submission not written.")

Detected score cols | Baseline: ['score_human', 'score_fusion', 'score_ai'] | mDeBERTa: ['logit_human', 'logit_fusion', 'logit_ai']

--- Validation Fold 0 Comparison (Nested CV, no leakage) ---
                          model  macro_f1   f1_H   f1_F   f1_A
             1. TF-IDF baseline    0.8883 0.8336 0.8451 0.9862
          2. mDeBERTa raw (ref)    0.9351 0.8933 0.9155 0.9966
 3. mDeBERTa + multipliers (CV)    0.9361 0.8967 0.9151 0.9966
4. mDeBERTa + TF-IDF blend (CV)    0.9544 0.9261 0.9396 0.9974
    5. Blend + multipliers (CV)    0.9543 0.9265 0.9391 0.9974

Mean blend multipliers [H, F, A]: [1.424 1.168 1.   ]

Saved submission_blend_mdeb_tfidf.csv (best: 4. mDeBERTa + TF-IDF blend (CV), +0.0193 vs ref)
label
human     25.44
fusion    23.19
ai        51.37
Name: proportion, dtype: float64
